# Week 3: Quantum Computing Workflow — The Qiskit Pattern
## Map → Optimize → Execute → Post-process

**Module:** [Development Workflow / Intro to Patterns](https://quantum.cloud.ibm.com/docs/en/guides/intro-to-patterns)

In this notebook we study the **Qiskit Pattern** — the canonical four-step development workflow for running any quantum algorithm on real (or simulated) hardware:

| Step | What happens |
|------|--------------|
| **1. Map** | Translate a classical problem into quantum circuits and operators |
| **2. Optimize** | Transpile abstract circuits to ISA circuits for target hardware |
| **3. Execute** | Run ISA circuits using Sampler or Estimator primitives |
| **4. Post-process** | Interpret, visualize, and error-mitigate the results |

We walk through **two worked examples** that each follow all four steps:
- **Example A:** A Bell state (entanglement) experiment — using Sampler
- **Example B:** Expectation value of a Pauli observable — using Estimator

> **Note:** We use `AerSimulator` for all experiments — no IBM Quantum account required.

---
## SECTION 1: Setup & Imports

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit import ParameterVector
from qiskit_aer import AerSimulator
from qiskit.primitives import BackendSamplerV2, BackendEstimatorV2
import numpy as np
import matplotlib.pyplot as plt

backend = AerSimulator()
print(f'Using backend: {backend.name}')

---
## SECTION 2: Overview — The Qiskit Pattern

The **Qiskit Pattern** is the standard development workflow recommended by IBM Quantum for any quantum computing task:

```
 ┌──────────────────────────────────────────────────────────┐
 │                    QISKIT PATTERN                        │
 │                                                          │
 │  Classical  ┌───────┐  Abstract  ┌──────────┐           │
 │   Problem ─►│  MAP  │─►Circuits ─►│ OPTIMIZE │           │
 │             └───────┘            └────┬─────┘           │
 │                                       │ ISA Circuits     │
 │  Interpreted  ┌──────────────┐        ▼                  │
 │   Results  ◄──│ POST-PROCESS │◄─── ┌─────────┐          │
 │               └──────────────┘     │ EXECUTE │          │
 │                                    └─────────┘          │
 └──────────────────────────────────────────────────────────┘
```

---
## SECTION 3: Example A — Bell State (Sampler Primitive)

### What is a Bell State?

A **Bell state** is a maximally entangled two-qubit state:

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}} (|00\rangle + |11\rangle)$$

Measuring one qubit instantly determines the other — this is quantum entanglement.

### STEP 1 — MAP: Translate to Quantum Circuit

**Mapping:** H gate on qubit 0 → superposition; CNOT → entanglement

$$|00\rangle \xrightarrow{H \otimes I} \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)|0\rangle \xrightarrow{CNOT} \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

In [ ]:
# STEP 1 — MAP: Build the abstract Bell state circuit
bell = QuantumCircuit(2)
bell.h(0)       # Hadamard gate: creates superposition
bell.cx(0, 1)   # CNOT: entangles the two qubits
bell.measure_all()

print('=== Abstract Bell Circuit ===')
print(f'Gates: {bell.count_ops()}')
print(f'Depth: {bell.depth()}')
bell.draw('mpl')

### STEP 2 — OPTIMIZE: Transpile to ISA Circuit

Real hardware only understands **basis gates** (e.g., `rz`, `sx`, `cx`). The transpiler:
- Decomposes abstract gates into basis gates
- Respects the hardware's **coupling map** (qubit connectivity)
- Minimizes gate count to reduce errors

> **Only ISA circuits can run on IBM quantum hardware.**

In [ ]:
# STEP 2 — OPTIMIZE: Transpile to ISA circuit
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
bell_isa = pm.run(bell)

print('=== Before Transpilation ===')
print(f'  Gates: {bell.count_ops()}, Depth: {bell.depth()}')
print()
print('=== After Transpilation (ISA) ===')
print(f'  Gates: {bell_isa.count_ops()}, Depth: {bell_isa.depth()}')
print(f'  Basis gates used: {set(bell_isa.count_ops().keys())}')
bell_isa.draw('mpl', fold=-1)

### STEP 3 — EXECUTE: Run with Sampler

The **Sampler** primitive returns per-shot bitstrings. For a Bell state, we expect only `|00⟩` and `|11⟩`.

In [ ]:
# STEP 3 — EXECUTE: Sampler primitive
sampler = BackendSamplerV2(backend=backend)
job = sampler.run([bell_isa], shots=1024)
result = job.result()

counts = result[0].data.meas.get_counts()
total = sum(counts.values())

print('=== Sampler Results ===')
print(f'Counts: {counts}')
print()
print('Probabilities:')
for bs, c in sorted(counts.items()):
    print(f'  |{bs}⟩: {c}/{total} = {c/total:.3f}')
print()
print('Expected: ONLY |00⟩ and |11⟩ — entanglement prevents |01⟩ and |10⟩')

### STEP 4 — POST-PROCESS: Visualize and Interpret

In [ ]:
# STEP 4 — POST-PROCESS: Visualize the Bell state results
all_states = ['00', '01', '10', '11']
probs = [counts.get(s, 0) / total for s in all_states]
colors = ['#4CAF50' if s in ['00', '11'] else '#E0E0E0' for s in all_states]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(all_states, probs, color=colors, edgecolor='black')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ideal 50%')
ax.set_title('Bell State Measurement Distribution\n(green = expected, gray = should be ~0)', fontsize=12)
ax.set_xlabel('Bitstring')
ax.set_ylabel('Probability')
ax.set_ylim(0, 0.75)
ax.legend()
for i, (s, p) in enumerate(zip(all_states, probs)):
    if p > 0.01:
        ax.text(i, p + 0.02, f'{p:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

corr = (counts.get('00',0) + counts.get('11',0)) / total
print(f'Correlation probability (|00⟩ + |11⟩): {corr:.4f}')
print('Near-perfect correlation confirms quantum entanglement!')

---
## SECTION 4: Example B — Expectation Value (Estimator Primitive)

In algorithms like VQE, you need the **expectation value of a Hamiltonian**:

$$\langle E \rangle = \langle \psi(\theta) | H | \psi(\theta) \rangle$$

We use:
- **Ansatz circuit** $|\psi(\theta)\rangle$: a parameterized 2-qubit circuit
- **Hamiltonian:** $H = ZZ + 0.5 \cdot XI$

### STEP 1 — MAP: Define Hamiltonian and Ansatz

In [ ]:
# STEP 1 — MAP: Observable and parameterized circuit
hamiltonian = SparsePauliOp.from_list([
    ('ZZ', 1.0),
    ('XI', 0.5),
])
print('Hamiltonian:', hamiltonian)

theta = ParameterVector('θ', length=4)
ansatz = QuantumCircuit(2)
ansatz.ry(theta[0], 0)
ansatz.ry(theta[1], 1)
ansatz.cx(0, 1)
ansatz.ry(theta[2], 0)
ansatz.ry(theta[3], 1)

print(f'Ansatz parameters: {list(ansatz.parameters)}')
ansatz.draw('mpl')

### STEP 2 — OPTIMIZE: Transpile and Apply Layout

After transpilation, the **observable** must be re-indexed to match the physical qubit layout using `.apply_layout()`.

In [ ]:
# STEP 2 — OPTIMIZE
pm2 = generate_preset_pass_manager(target=backend.target, optimization_level=3)
ansatz_isa = pm2.run(ansatz)

# CRITICAL: map the observable to the physical qubit layout
hamiltonian_isa = hamiltonian.apply_layout(layout=ansatz_isa.layout)

print(f'Abstract circuit gates: {ansatz.count_ops()}')
print(f'ISA circuit gates: {ansatz_isa.count_ops()}')
print()
print('Hamiltonian after layout mapping:')
print(hamiltonian_isa)

### STEP 3 — EXECUTE: Run with Estimator

In [ ]:
# STEP 3 — EXECUTE: Estimator primitive
theta_values = [
    [0.0, 0.0, 0.0, 0.0],
    [np.pi/4, np.pi/4, np.pi/4, np.pi/4],
    [np.pi/2, 0.0, np.pi/2, 0.0],
    [np.pi, np.pi, np.pi, np.pi],
]

estimator = BackendEstimatorV2(backend=backend)
pubs = [(ansatz_isa, hamiltonian_isa, params) for params in theta_values]
job_ev = estimator.run(pubs)
res_ev = job_ev.result()

evs = [float(res_ev[i].data.evs) for i in range(len(theta_values))]

print('=== Estimator Results: ⟨H⟩ ===')
labels = ['[0,0,0,0]', '[π/4 each]', '[π/2,0,π/2,0]', '[π each]']
for label, ev in zip(labels, evs):
    print(f'  θ={label:<18} → ⟨H⟩ = {ev:.4f}')

### STEP 4 — POST-PROCESS: Visualize Energy Landscape

In [ ]:
# STEP 4 — POST-PROCESS: Energy landscape visualization
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2196F3' if ev >= 0 else '#F44336' for ev in evs]
bars = ax.bar(labels, evs, color=colors, edgecolor='black', alpha=0.85)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Expectation Value ⟨ZZ + 0.5·XI⟩ vs Parameter Configuration', fontsize=12)
ax.set_ylabel('⟨H⟩ (Energy)')
ax.set_xlabel('Parameter Configuration')
for bar, ev in zip(bars, evs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{ev:.3f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Minimum ⟨H⟩ = {min(evs):.4f} at θ={labels[evs.index(min(evs))]}')
print('In VQE, an optimizer would search for the parameter set that minimizes ⟨H⟩.')

---
## SECTION 5: Sampler vs Estimator — Side-by-Side

| Feature | **Sampler** | **Estimator** |
|---------|------------|---------------|
| **Output** | Bitstring counts | Expectation values |
| **Needs measurement gate** | Yes | No |
| **Use case** | Probability distributions | Energy/cost functions |
| **Example algorithms** | Grover's, QFT | VQE, QAOA, QEC |
| **Execution modes** | Batch, Session | Batch, Session |

In [ ]:
# Sampler vs Estimator on Bell state
# Estimator needs the circuit WITHOUT measure_all()
bell_no_meas = QuantumCircuit(2)
bell_no_meas.h(0)
bell_no_meas.cx(0, 1)
bell_no_meas_isa = pm.run(bell_no_meas)

obs_ZZ = SparsePauliOp.from_list([('ZZ', 1.0)]).apply_layout(bell_no_meas_isa.layout)
obs_XX = SparsePauliOp.from_list([('XX', 1.0)]).apply_layout(bell_no_meas_isa.layout)

est2 = BackendEstimatorV2(backend=backend)
job_e = est2.run([(bell_no_meas_isa, obs_ZZ), (bell_no_meas_isa, obs_XX)])
res_e = job_e.result()

print('=== Estimator (Bell State) ===')
print(f'⟨ZZ⟩ = {float(res_e[0].data.evs):.4f}  (Expected: +1 — qubits always agree)')
print(f'⟨XX⟩ = {float(res_e[1].data.evs):.4f}  (Expected: +1 — Bell state is XX eigenstate)')
print()
print('=== Sampler (Bell State) ===')
samp2 = BackendSamplerV2(backend=backend)
job_s = samp2.run([bell_isa], shots=1024)
c2 = job_s.result()[0].data.meas.get_counts()
print(f'Counts: {c2}')

---
## SECTION 6: Transpiler Optimization Levels

In [ ]:
# Compare transpilation optimization levels on a complex circuit
complex_qc = QuantumCircuit(3)
complex_qc.h(0); complex_qc.h(1); complex_qc.h(2)
complex_qc.cx(0, 1); complex_qc.cx(1, 2); complex_qc.cx(0, 2)
complex_qc.ry(np.pi/3, 0); complex_qc.rz(np.pi/4, 1)
complex_qc.cx(0, 1); complex_qc.h(0)
complex_qc.measure_all()

print('=== Transpilation Level Comparison ===')
print(f'{"Level":<8} {"Gates":<10} {"Depth":<10}')
print('-' * 30)
gate_counts = []
for level in [0, 1, 2, 3]:
    pm_l = generate_preset_pass_manager(target=backend.target, optimization_level=level)
    isa_l = pm_l.run(complex_qc)
    n_gates = sum(isa_l.count_ops().values())
    gate_counts.append(n_gates)
    print(f'{level:<8} {n_gates:<10} {isa_l.depth():<10}')
print()
reduction = (gate_counts[0] - gate_counts[3]) / gate_counts[0] * 100
print(f'Level 3 reduces gate count by ~{reduction:.1f}% vs Level 0')
print('Fewer gates → fewer hardware errors → better results!')

---
## SECTION 7: Full Qiskit Pattern — GHZ State

**GHZ (Greenberger–Horne–Zeilinger) state:** The 3-qubit generalization of Bell state:

$$|GHZ\rangle = \frac{1}{\sqrt{2}}(|000\rangle + |111\rangle)$$

Measuring any one qubit instantly determines the state of all three.

In [ ]:
# COMPLETE QISKIT PATTERN — GHZ State

# ── STEP 1: MAP ──────────────────────────────────
print('[STEP 1 — MAP] Building GHZ circuit...')
ghz = QuantumCircuit(3)
ghz.h(0)
ghz.cx(0, 1)
ghz.cx(1, 2)
ghz.measure_all()
display(ghz.draw('mpl'))

# ── STEP 2: OPTIMIZE ─────────────────────────────
print('[STEP 2 — OPTIMIZE] Transpiling to ISA...')
pm_ghz = generate_preset_pass_manager(target=backend.target, optimization_level=3)
ghz_isa = pm_ghz.run(ghz)
print(f'  ISA gates: {ghz_isa.count_ops()}, Depth: {ghz_isa.depth()}')

# ── STEP 3: EXECUTE ──────────────────────────────
print('[STEP 3 — EXECUTE] Running (2048 shots)...')
samp_ghz = BackendSamplerV2(backend=backend)
job_ghz = samp_ghz.run([ghz_isa], shots=2048)
res_ghz = job_ghz.result()
counts_ghz = res_ghz[0].data.meas.get_counts()
print(f'  Results: {counts_ghz}')

# ── STEP 4: POST-PROCESS ─────────────────────────
print('[STEP 4 — POST-PROCESS] Analyzing...')
total_ghz = sum(counts_ghz.values())
p000 = counts_ghz.get('000', 0) / total_ghz
p111 = counts_ghz.get('111', 0) / total_ghz
print(f'  P(|000⟩) = {p000:.4f}')
print(f'  P(|111⟩) = {p111:.4f}')
print(f'  GHZ fidelity ≈ {p000+p111:.4f} (should be ~1.0)')

In [ ]:
# GHZ: Final visualization
all_3q = [format(i,'03b') for i in range(8)]
probs_ghz = [counts_ghz.get(s,0)/total_ghz for s in all_3q]
colors_ghz = ['#4CAF50' if s in ['000','111'] else '#E0E0E0' for s in all_3q]

fig, ax = plt.subplots(figsize=(10,5))
ax.bar(all_3q, probs_ghz, color=colors_ghz, edgecolor='black')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Ideal 50%')
ax.set_title('GHZ State: 3-Qubit Measurement Distribution\n(green = expected, gray = should be ~0)', fontsize=12)
ax.set_xlabel('Bitstring |q₂q₁q₀⟩'); ax.set_ylabel('Probability')
ax.set_ylim(0, 0.75); ax.legend()
for i,(s,p) in enumerate(zip(all_3q,probs_ghz)):
    if p > 0.01:
        ax.text(i, p+0.02, f'{p:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()
print('✓ Only |000⟩ and |111⟩ appear — maximal 3-qubit entanglement confirmed!')

---
## SECTION 8: Summary & Key Takeaways

### The Qiskit Pattern — Quick Reference

| Step | Goal | Key Tools |
|------|------|-----------|
| **1. Map** | Classical problem → quantum circuits + operators | `QuantumCircuit`, `SparsePauliOp`, `ParameterVector` |
| **2. Optimize** | Abstract → ISA circuit | `generate_preset_pass_manager`, `.apply_layout()` |
| **3. Execute** | Run on backend | `BackendSamplerV2`, `BackendEstimatorV2` |
| **4. Post-process** | Interpret results | `.get_counts()`, `.evs`, `plot_histogram` |

### Critical Concepts

1. **ISA circuits are mandatory** for real IBM hardware — the transpiler handles this conversion.
2. **Sampler** → bitstring counts/probabilities; **Estimator** → expectation values.
3. **Observable layout must match** the physical qubit layout after transpilation.
4. **Higher optimization levels** reduce gate count → fewer hardware errors.
5. The pattern is **composable and modular** — each step can be independently improved.

### Questions for Reflection

| # | Question |
|---|----------|
| 1 | Why can't we run an abstract circuit directly on IBM quantum hardware? |
| 2 | What is the difference between a Session and a Batch execution mode? |
| 3 | When would you choose Estimator over Sampler? |
| 4 | What does `.apply_layout()` do and why is it necessary? |
| 5 | Why does higher optimization level reduce gates but take longer to compile? |

### Answers

1. Hardware only understands its **basis gates** — abstract gates must be decomposed. Also, qubits must be connected (coupling map) to perform 2-qubit operations.

2. **Session** reserves a QPU connection for iterative jobs (ideal for VQE/QAOA). **Batch** submits multiple independent jobs in parallel (ideal for parameter sweeps).

3. Use **Estimator** for energy/cost functions (VQE, QAOA, quantum chemistry). Use **Sampler** for full probability distributions (Grover's, QPE).

4. Transpilation remaps logical qubits to physical qubits. `.apply_layout()` re-indexes the observable's Pauli terms to match the physical qubit indices, ensuring the Estimator measures the right qubits.

5. Higher optimization applies more compiler passes (gate cancellations, routing optimization) which require more classical computation — but save quantum resources and reduce error rates.